## Import librairies

In [46]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


## Columns definitions

In [47]:
# 0-based indices of the 14 used attributes within the 76-token patient block
USED_INDICES = [2, 3, 8, 9, 11, 15, 18, 31, 37, 39, 40, 43, 50, 57]
 
COLUMNS = [
    "age",       # age in years
    "sex",       # 1 = male; 0 = female
    "cp",        # chest pain type (1–4)
    "trestbps",  # resting blood pressure (mm Hg)
    "chol",      # serum cholesterol (mg/dl)
    "fbs",       # fasting blood sugar > 120 mg/dl (1 = true; 0 = false)
    "restecg",   # resting ECG results (0, 1, 2)
    "thalach",   # maximum heart rate achieved
    "exang",     # exercise induced angina (1 = yes; 0 = no)
    "oldpeak",   # ST depression induced by exercise relative to rest
    "slope",     # slope of peak exercise ST segment (1, 2, 3)
    "ca",        # number of major vessels coloured by fluoroscopy (0–3)
    "thal",      # thal: 3 = normal; 6 = fixed defect; 7 = reversable defect
    "num",       # target: diagnosis of heart disease (0–4)
]
 
CATEGORICAL_COLUMNS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
NUMERICAL_COLUMNS   = ["age", "trestbps", "chol", "thalach", "oldpeak"]
 


## Load raw-data

In [48]:
DATA_FILE = "../raw/heart-data/cleveland.data"   
 
def load_data(path: str) -> pd.DataFrame:
    """
    The raw file stores each patient's 76 attributes as whitespace-separated
    tokens spread across multiple lines.  We read the whole file as a flat
    token list and slice out the 14 used attributes for each of the 303 patients.
    """
    with open(path, encoding="latin-1") as f:
        tokens = f.read().split()
 
    n_patients = 303
    n_attrs    = 76
 
    rows = []
    for i in range(n_patients):
        start = i * n_attrs
        block = tokens[start : start + n_attrs]
        if len(block) == n_attrs:
            rows.append([block[j] for j in USED_INDICES])
 
    return pd.DataFrame(rows, columns=COLUMNS)
 
data = load_data(DATA_FILE)
print(f"Loaded shape: {data.shape}")

Loaded shape: (297, 14)


## Basic cleaning

In [49]:
# Missing values are encoded as -9 in this dataset
data.replace("-9", np.nan, inplace=True)
 
# Convert all columns to numeric (non-parseable tokens become NaN)
for col in COLUMNS:
    data[col] = pd.to_numeric(data[col], errors="coerce")
 
# Drop the few corrupt rows where the target is outside the valid 0–4 range
data = data[data["num"].isin([0, 1, 2, 3, 4])]
 
# Binarise the target: 0 = no disease, 1 = disease present (values 1–4)
data["num"] = (data["num"] > 0).astype(int)
 
print(f"After cleaning: {data.shape}")
print(f"Target distribution:\n{data['num'].value_counts().sort_index()}")


After cleaning: (286, 14)
Target distribution:
num
0    157
1    129
Name: count, dtype: int64


## Traint/Test split

In [50]:
X = data.drop("num", axis=1)
y = data["num"].rename("target")
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Missing values

In [51]:
# Categorical columns → impute with the most frequent value (from train only)
for col in CATEGORICAL_COLUMNS:
    most_frequent = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(most_frequent)
    X_test[col]  = X_test[col].fillna(most_frequent)
 
# Numerical columns → impute with the mean (from train only)
for col in NUMERICAL_COLUMNS:
    mean_val = X_train[col].mean()
    X_train[col] = X_train[col].fillna(mean_val)
    X_test[col]  = X_test[col].fillna(mean_val)

## Normalization

In [52]:
# Export the imputed train/test split before normalization for TabLLM.
serialization_output_dir = Path("../pre-processed/serialization")
serialization_output_dir.mkdir(parents=True, exist_ok=True)
pd.concat([X_train.copy(), y_train], axis=1).to_csv(
    serialization_output_dir / "heart_train.csv", index=False
)
pd.concat([X_test.copy(), y_test], axis=1).to_csv(
    serialization_output_dir / "heart_test.csv", index=False
)

# Fit normalization only on training data, then apply it to test data.
scaler = StandardScaler()
X_train[NUMERICAL_COLUMNS] = scaler.fit_transform(X_train[NUMERICAL_COLUMNS])
X_test[NUMERICAL_COLUMNS]  = scaler.transform(X_test[NUMERICAL_COLUMNS])


## Export

In [53]:
train_data = pd.concat([X_train, y_train], axis=1)
test_data  = pd.concat([X_test,  y_test],  axis=1)

disease_name = "heart"
  
print(f"\nExported → {disease_name}_train.csv {train_data.shape}, {disease_name}_test.csv {test_data.shape}")
print(f"Target balance (train): {y_train.value_counts().to_dict()}")



Exported → heart_train.csv (228, 14), heart_test.csv (58, 14)
Target balance (train): {0: 125, 1: 103}


In [54]:
disease_name = "heart"

output_dir = Path("../pre-processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_data.to_csv(output_dir / f"{disease_name}_train.csv", index=False)
test_data.to_csv(output_dir / f"{disease_name}_test.csv", index=False)